# Stock Markets Analytics Zoomcamp 2025 - Module 01
## Introduction to Stock Markets Analytics and Data Sources

**Course by Ivan (Business Intelligence Analyst at Google) and PythonInvest.com**

### Course Resources
- 📹 [YouTube Recording](https://youtu.be/2zlv2nU7g58)
- 💻 [GitHub Repository](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp)
- 📊 [Presentation Slides](https://docs.google.com/presentation/d/e/2PACX-1vR_vfIYCpGhgsR_jef9uo5YdKbg68LGO6pZR5kRSrxDTHNRujKgPb7r9K1U1SM9yOFJlC7OoDAAjKHG/pub?start=false&loop=false&delayms=10000&slide=id.g2c7aa6c5021_0_22)

### Module Overview
In this module, we will:
1. Understand data-driven decision making for investments
2. Learn about macroeconomic indicators and their importance
3. Explore various data sources for stock market analysis
4. Set up our development environment using Google Colab
5. Practice extracting and visualizing financial data

### Important Note
⚠️ **This is not investment advice!** All content is for educational purposes only. You should use this knowledge to learn about financial markets analysis, not for making actual investment decisions without proper research and risk assessment.

## Course Objectives and Expected Outcomes

### Seven Levels of Complexity

1. **Level 1 - Basic Skills**: Replicate notebooks, run code, gain Python/analytics/finance skills
2. **Level 2 - Start Trading**: Register with broker, transfer funds, practice small trades
3. **Level 3 - Competition**: Complete all homework assignments and compete on the leaderboard
4. **Level 4 - Portfolio Project**: Create your own project that generates trade recommendations
5. **Level 5 - Real Implementation**: Start making investment decisions with your code
6. **Level 6 - Consistent Profits**: Achieve monthly/quarterly returns on paper or real money
7. **Level 7 - Long-term Success**: Sustained profitability over years with constant model improvement

### What You'll Learn
- Analyzing and predicting stock market trends
- Developing trading strategies and algorithms
- Creating personal investment portfolios
- Understanding financial markets for personal interest
- Building financial analysis tools and dashboards

## Google Colab - Our Development Environment

### Why Google Colab?
Google Colab is the recommended environment for this course due to:

✅ **Advantages:**
- **Low barriers to start**: No installation required, runs in browser
- **Pre-installed libraries**: Python, data science, and ML libraries ready to use
- **Free GPU access**: Useful for training complex models with millions of records
- **Easy sharing and saving**: Cloud-based with automatic saving
- **Dynamic visualizations**: Interactive graphs in the notebook

❗ **Limitations:**
- Sessions may disconnect after ~1 hour of inactivity
- May need to re-upload files (or use Google Drive integration)
- Free tier has usage limits (usually sufficient for course)

### Alternative Setup
You can also use:
- Local Jupyter Notebooks with Anaconda
- VS Code with Python extension
- Any Python IDE of your choice

📚 **Resources:**
- [Google Colab](https://colab.research.google.com/)
- [Python Environment Setup](https://pythoninvest.com/long-read/python-environment)
- [Colab Pricing](https://colab.research.google.com/signup)

## 0) Environment Setup and Imports

Let's start by setting up our environment and importing the necessary libraries.

In [ ]:
# Install the main library YFinance
# YFinance is our primary tool for downloading stock market data
!pip install yfinance

In [ ]:
# IMPORTS
import numpy as np
import pandas as pd

# Financial Data Sources
import yfinance as yf
import pandas_datareader as pdr

# Data Visualization
import plotly.graph_objs as go
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Utilities
import time
from datetime import date

# For web scraping (we'll use later)
import requests
from bs4 import BeautifulSoup

print("All libraries imported successfully!")

# 1) Understanding Data-Driven Decisions

## Making Informed Investment Choices

Before we dive into the code, let's understand the key concepts that drive investment decisions:

### 💡 Key Investment Concepts

1. **Rate of Return**: The gain or loss on an investment over a specified period, expressed as a percentage
2. **Risk vs Reward**: Higher potential returns usually come with higher risk
3. **Inflation**: The rate at which the general level of prices increases, eroding purchasing power
4. **Benchmark**: A standard against which the performance of an investment can be measured

### 📊 What We'll Analyze

We'll pull data for several key economic indicators:
- **GDP (Gross Domestic Product)**: Measures economic growth
- **CPI (Consumer Price Index)**: Measures inflation
- **Interest Rates**: Fed funds rate and treasury yields
- **Stock Market Indices**: S&P 500 as a benchmark

Let's start by setting up our date range for analysis:

In [ ]:
# Set up date range for our analysis
# We'll look at up to 70 years of historical data where available
end = date.today()
print(f'Current date: {end}')
print(f'Year = {end.year}; month = {end.month}; day = {end.day}')

start = date(year=end.year-70, month=end.month, day=end.day)
print(f'\nPeriod for analysis: {start} to {end}')
print(f'That\'s {(end - start).days // 365} years of data!')

## 1.1) GDP - Understanding Economic Growth

### What is GDP?
**Real Potential GDP** is the CBO's estimate of the output the economy could produce if its capital and labor resources were used at a high rate. This data is adjusted to remove the effects of inflation.

### Why GDP Matters for Investors
- GDP growth indicates a healthy, expanding economy
- Companies tend to perform better in growing economies
- The prediction has been stable: +2-2.3% YoY for the last 5 years

### Data Source
We'll use FRED (Federal Reserve Economic Data) to get GDP data.

📚 **Resources:**
- [FRED GDP Series](https://fred.stlouisfed.org/series/GDPPOT)
- [FRED Documentation](https://data.nasdaq.com/data/FRED-federal-reserve-economic-data/documentation)

In [ ]:
# Real Potential Gross Domestic Product (GDPPOT), Billions of Chained 2012 Dollars, QUARTERLY
# https://fred.stlouisfed.org/series/GDPPOT
gdppot = pdr.DataReader("GDPPOT", "fred", start=start)
print(f"GDP data loaded: {len(gdppot)} quarters of data")
print(f"Date range: {gdppot.index[0]} to {gdppot.index[-1]}")

In [ ]:
# Calculate growth rates
# Year-over-Year (YoY): Compare to same quarter last year (4 quarters ago)
# Quarter-over-Quarter (QoQ): Compare to previous quarter
gdppot['gdppot_us_yoy'] = gdppot.GDPPOT/gdppot.GDPPOT.shift(4)-1
gdppot['gdppot_us_qoq'] = gdppot.GDPPOT/gdppot.GDPPOT.shift(1)-1

# Display recent data
print("Recent GDP Data and Growth Rates:")
gdppot.tail(15)

In [ ]:
# Visualize GDP and Growth Rate
fig, ax = plt.subplots(figsize=(20, 6))
plt.grid(True)

# Plotting area under US potential GDP curve
ax.fill_between(gdppot.index, gdppot.GDPPOT, color="red", alpha=0.3, label="US Potential GDP")

# Creating a secondary y-axis for GDP growth percentage
ax2 = ax.twinx()
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.plot(gdppot.gdppot_us_yoy, color="blue", marker="o", label="US Potential GDP Growth, % Y/Y")

# Setting labels and title
ax.set_xlabel("Date", fontsize=14)
ax.set_ylabel("US Potential GDP, $b", color="red", fontsize=14)
ax2.set_ylabel("US Potential GDP Growth, % Y/Y", color="blue", fontsize=14)
ax.set_title("US Potential GDP and Year-over-Year Growth Rate", fontsize=16)

# Adding legend
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.show()

## 1.2) Inflation - Core CPI

### Do I Save Faster than Inflation?

The **"Consumer Price Index for All Urban Consumers: All Items Less Food & Energy"** is an aggregate of prices paid by urban consumers for a typical basket of goods, excluding food and energy. This measurement, known as **"Core CPI"**, is widely used by economists because food and energy have very volatile prices.

### Current Inflation Status
- **Core CPI** declined from 3.7% in Feb 2024 to 2.8% in Mar 2025
- This means you need to earn at least 2.8% on your savings just to maintain purchasing power!

### Why This Matters
- If your savings earn less than inflation, you're losing money in real terms
- Investment returns should be evaluated after adjusting for inflation
- Different countries have different inflation rates (e.g., Ireland: 2.2%, US: 2.8%)

📚 **Resources:**
- [FRED CPI Series](https://fred.stlouisfed.org/series/CPILFESL)
- [10-Year Inflation Expectations](https://fred.stlouisfed.org/series/T10YIE)

In [ ]:
# "Core CPI index", MONTHLY
# https://fred.stlouisfed.org/series/CPILFESL
cpilfesl = pdr.DataReader("CPILFESL", "fred", start=start)
print(f"CPI data loaded: {len(cpilfesl)} months of data")
print(f"Date range: {cpilfesl.index[0]} to {cpilfesl.index[-1]}")

In [ ]:
# Calculate inflation rates
# Year-over-Year inflation: Compare to same month last year
# Month-over-Month inflation: Compare to previous month
cpilfesl['cpi_core_yoy'] = cpilfesl.CPILFESL/cpilfesl.CPILFESL.shift(12)-1
cpilfesl['cpi_core_mom'] = cpilfesl.CPILFESL/cpilfesl.CPILFESL.shift(1)-1

print("Recent Core CPI Data and Inflation Rates:")
print(f"Current YoY Core Inflation: {cpilfesl['cpi_core_yoy'].iloc[-1]:.2%}")
cpilfesl.tail(13)

In [ ]:
# Visualize CPI and Inflation Rate
fig, ax = plt.subplots(figsize=(20, 6))
plt.grid(True)

# Plotting area under CPI
ax.fill_between(cpilfesl.index, cpilfesl.CPILFESL, color="red", alpha=0.3, label="Core CPI index (monthly)")

# Creating a secondary y-axis for CPI growth percentage
ax2 = ax.twinx()
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.plot(cpilfesl.cpi_core_yoy, color="blue", marker="o", label="Core CPI index (monthly) Growth, % Y/Y")

# Add a horizontal line at 2% (Fed's target)
ax2.axhline(y=0.02, color='green', linestyle='--', alpha=0.7, label='Fed Target (2%)')

# Setting labels and title
ax.set_xlabel("Date", fontsize=14)
ax.set_ylabel("Core CPI index (monthly)", color="red", fontsize=14)
ax2.set_ylabel("Core CPI index Growth, % Y/Y", color="blue", fontsize=14)
ax.set_title("Core CPI and Year-over-Year Inflation Rate", fontsize=16)

# Adding legend
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.show()

## 💰 Saving Options - Where to Keep Your Money

### Saving Money with Cash Interest (2025 Rates)

Let's compare different options for saving money and their returns:

| Strategy | Interest Rate | Pros | Cons |
|----------|--------------|------|------|
| **Do Nothing** | 0% | Full liquidity | Losing to inflation |
| **Bank Savings** | 2-3% | Money protection scheme | Often below inflation |
| **Money Service Business (Wise)** | 2.24% EUR, 4.08% USD | Easy transfers | Not always protected |
| **Broker Accounts** | 0-4.1% | Can invest easily | Conditions apply |

### Specific Examples (April 2025):
- 🏦 **Traditional Banks**: 2-3% in EUR (Ireland)
- 💳 **Wise**: 2.24% EUR (↓ from 3.67%), 4.08% USD (↓ from 5.05%)
- 📈 **Brokers**:
  - DEGIRO: 0% on uninvested cash
  - Trade Republic: 3-3.25% EUR (↓ from 4%)
  - Interactive Brokers: 4.1% USD (↓ from 4.83%), 2.7% EUR

### Key Takeaway
With US inflation at 2.8% and Irish inflation at 2.2%, you need to earn more than these rates just to maintain your purchasing power!

## 1.3) Interest Rates - The Foundation of Investment Returns

### Fed Funds Rate - The Most Important Rate

The **Federal Funds Rate** is the interest rate at which banks trade federal funds with each other overnight. It's the foundation for all other interest rates in the economy.

### Current Rate Environment (2025)
- Fed Funds Rate: 4.33% (↓ from 5.33% a year ago)
- Risk-free rate (3-month T-bill): 4.21%
- 10-Year Treasury: 4.29%
- Markets expect 2 rate cuts in 2025

### Investment Implications
- **Bonds**: Corporate bonds yield 5-7%+ (risk premium over treasuries)
  - AAA-rated: ~5.3%
  - BAA-rated: ~5.9%
- **Stocks**: Should target returns well above risk-free rate

📚 **Resources:**
- [FRED Fed Funds Rate](https://fred.stlouisfed.org/series/FEDFUNDS)
- [Treasury Yield Curve](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve&field_tdr_date_value=2025)

In [ ]:
# Fed rate https://fred.stlouisfed.org/series/FEDFUNDS
fedfunds = pdr.DataReader("FEDFUNDS", "fred", start=start)
print(f"Current Fed Funds Rate: {fedfunds['FEDFUNDS'].iloc[-1]:.2f}%")
fedfunds.tail(10)

In [ ]:
# Visualize Fed Funds Rate History
fig, ax = plt.subplots(figsize=(20, 6))
plt.grid(True)

# Format y-axis as percentage
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.plot(fedfunds.index, fedfunds.FEDFUNDS/100, marker="o", label="Fed Funds Rate")

# Add horizontal lines for reference
ax.axhline(y=0.02, color='green', linestyle='--', alpha=0.5, label='2% (Low rates)')
ax.axhline(y=0.05, color='orange', linestyle='--', alpha=0.5, label='5% (Moderate rates)')

# Setting labels and title
ax.set_xlabel("Date", fontsize=14)
ax.set_ylabel("Fed Funds Rate", color="blue", fontsize=14)
ax.set_title("Federal Funds Rate History - The Foundation of All Interest Rates", fontsize=16)

# Adding legend
ax.legend(loc='upper left')

plt.show()

In [ ]:
# Get Treasury Rates for comparison
# 1-Month Treasury Rate
dgs1 = pdr.DataReader("DGS1", "fred", start=start)
print(f"Current 1-Month Treasury Rate: {dgs1['DGS1'].iloc[-1]:.2f}%")
dgs1.tail()

### Treasury Yield Curve

The yield curve shows interest rates across different maturities. Normal curves slope upward (longer = higher yield), but sometimes invert (recession signal).

Other Treasury rates available:
* [2-Year](https://fred.stlouisfed.org/series/DGS2)
* [3-Year](https://fred.stlouisfed.org/series/DGS3)
* [5-Year](https://fred.stlouisfed.org/series/DGS5)
* [10-Year](https://fred.stlouisfed.org/series/DGS10)
* [30-Year](https://fred.stlouisfed.org/series/DGS30)

In [ ]:
# 5-Year Treasury Rate for comparison
dgs5 = pdr.DataReader("DGS5", "fred", start=start)
print(f"Current 5-Year Treasury Rate: {dgs5['DGS5'].iloc[-1]:.2f}%")
dgs5.tail()

## 1.4) S&P 500 - The Ultimate Stock Market Benchmark

### Why S&P 500?
The S&P 500 Index tracks 500 leading U.S. companies, offering a snapshot of the country's stock market health. It's the go-to benchmark for investors.

### Historical Performance
- **Average annual return (1990-2025)**: 10.31%
- **Inflation-adjusted return**: 7.54% per year
- **2024 performance**: +23.31%
- **2025 YTD (as of April)**: -6.06%

### Key Insights
- Individual stocks can have much wider return ranges (both positive and negative)
- The index smooths out individual stock volatility
- Years can vary dramatically (from -40% to +30%+)

📚 **Resource**: [TradingView Stock Heatmap](https://www.tradingview.com/heatmap/stock/)

In [ ]:
# Download S&P 500 data
# Note: We use stooq for free S&P 500 data
spx_index = pdr.get_data_stooq('^SPX', start, end)
print(f"S&P 500 data loaded: {len(spx_index)} trading days")
spx_index.head()

In [ ]:
# Calculate returns
# Note: stooq data is in reverse order!
# 252 trading days in a typical year
spx_index['spx_dod'] = (spx_index.Close/spx_index.Close.shift(-1)-1)  # Daily return
spx_index['spx_qoq'] = (spx_index.Close/spx_index.Close.shift(-63)-1)  # Quarterly return
spx_index['spx_yoy'] = (spx_index.Close/spx_index.Close.shift(-252)-1)  # Annual return

In [ ]:
# Focus on data from 1990 onwards for cleaner visualization
spx_truncated = spx_index[spx_index.index>='1990-01-01']

In [ ]:
# Visualize S&P 500 absolute value and relative growth
fig, ax = plt.subplots(figsize=(20, 6))
plt.grid(True)

# Plotting area under S&P 500
ax.fill_between(spx_truncated.index, spx_truncated.Close, color="red", alpha=0.3, label="S&P 500 Absolute Value (Close price)")

# Creating a secondary y-axis for growth percentage
ax2 = ax.twinx()
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.plot(spx_truncated.spx_yoy, color="blue", label="Year-over-Year Growth (%)")

# Add reference lines
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax2.axhline(y=0.10, color='green', linestyle='--', alpha=0.5, label='10% (Long-term average)')

# Setting labels and title
ax.set_xlabel("Date", fontsize=14)
ax.set_ylabel("S&P 500 Absolute Value (Close price)", color="red", fontsize=14)
ax2.set_ylabel("Year-over-Year Growth (%)", color="blue", fontsize=14)
ax.set_title("S&P 500 Index: Absolute Value and Annual Returns", fontsize=16)

# Adding legend
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.show()

# Print summary statistics
print(f"\nS&P 500 Performance Summary (1990-present):")
print(f"Average Annual Return: {spx_truncated['spx_yoy'].mean():.2%}")
print(f"Best Year: {spx_truncated['spx_yoy'].max():.2%}")
print(f"Worst Year: {spx_truncated['spx_yoy'].min():.2%}")
print(f"Volatility (Std Dev): {spx_truncated['spx_yoy'].std():.2%}")

## 💡 Investment Strategies and Benchmarks

### Rule of 72 - How Quickly Can You Double Your Money?

The Rule of 72 is a simple way to determine how long an investment will take to double given a fixed annual rate of interest. By dividing 72 by the annual rate of return, you get the approximate number of years to double your investment.

| Annual Return | Years to Double | Example Investment |
|--------------|-----------------|--------------------|
| 2% | 36 years | Savings Account |
| 4% | 18 years | Bonds |
| 7% | 10 years | 60/40 Portfolio |
| 10% | 7.2 years | S&P 500 Average |
| 20% | 3.6 years | Top Hedge Funds |

### Long-term Success Benchmarks

**What defines a successful investment strategy?**

1. **Positive Returns**: Rolling positive returns over any 6-12 month period
2. **Beat Inflation**: Real returns after adjusting for inflation
3. **Beat Benchmarks**: Outperform relevant indices or risk-free rate

**Common Benchmarks:**
- **60/40 Portfolio**: ~7.1% nominal (5% real)
- **S&P 500**: ~10.5% nominal (6.3% real)
- **Warren Buffett**: ~20% CAGR over decades
- **Renaissance Technologies**: ~66% CAGR (before fees)

### Risk-Reward by Asset Class

Different asset classes offer different risk-reward profiles:

| Asset Class | Expected Return | Risk Level | Best For |
|-------------|----------------|------------|----------|
| Cash | 0-4% | Very Low | Emergency Fund |
| Bonds | 4-7% | Low | Stability |
| Real Estate | 8-12% | Medium | Diversification |
| Large-cap Stocks | 10-12% | Medium-High | Growth |
| Small-cap Stocks | 12-15% | High | Aggressive Growth |
| Emerging Markets | 15-20% | Very High | Risk Seekers |

### Your Personal Investment Framework

Consider these factors when developing your strategy:

1. **Risk Tolerance**: How much volatility can you handle?
2. **Time Horizon**: When do you need the money?
3. **Goals**: Capital preservation vs. growth?
4. **Age**: Generally take more risk when younger

**Sweet Spot**: 6-10% annual returns (net of inflation) is a reasonable target for most investors.

# 2) Data Sources for Stocks

## Overview of Financial Data Sources

Now that we understand the economic context, let's explore how to get data for individual stocks and other financial instruments.

### 📊 Types of Financial Data

1. **OHLCV Data**: Open, High, Low, Close, Volume - the foundation of technical analysis
2. **Technical Indicators**: Moving averages, RSI, MACD (covered in Module 2)
3. **Fundamental Data**: Financial statements, earnings, ratios
4. **Macroeconomic Data**: GDP, inflation, interest rates (already covered)
5. **Alternative Data**: News, social media, satellite imagery, web traffic

### 🔧 Our Main Tools

- **Yahoo Finance (yfinance)**: Free, reliable, good for daily data
- **FRED (pandas_datareader)**: Macroeconomic data
- **Paid APIs**: Polygon.io, Alpha Vantage for more detailed/real-time data
- **Web Scraping**: When APIs aren't available

## 2.1 OHLCV Data - Stock Market Indices

Let's start by downloading data for major stock market indices. These serve as benchmarks for overall market performance.

In [ ]:
# Download DAX Index (German stock market)
# DAX index (XETRA - XETRA Delayed Price. Currency in EUR)
# Web: https://finance.yahoo.com/quote/%5EGDAXI

ticker_obj = yf.Ticker("^GDAXI")
dax_daily = ticker_obj.history(start = start)

print(f"DAX Index data loaded: {len(dax_daily)} trading days")
print(f"Date range: {dax_daily.index[0]} to {dax_daily.index[-1]}")

In [ ]:
# Display recent DAX data
dax_daily.tail()

In [ ]:
# Calculate year-over-year growth
# Typically 252 trading days in a year
dax_daily['adj_close_last_year'] = dax_daily['Close'].shift(252)
dax_daily['yoy_growth'] = dax_daily['Close'] / dax_daily['adj_close_last_year'] - 1

print(f"Current DAX YoY Performance: {dax_daily['yoy_growth'].iloc[-1]:.2%}")

In [ ]:
# Simple visualization of DAX performance
dax_daily['Close'].plot(figsize=(15, 6), title="DAX Index Performance", ylabel="Index Value")
plt.grid(True)
plt.show()

In [ ]:
# Download S&P 500 data from Yahoo Finance
# Note: ^SPX is delayed 15 min, ^GSPC is real-time
ticker_obj = yf.Ticker("^GSPC")
snp500_daily = ticker_obj.history(start = start, interval = "1d")

print(f"S&P 500 data from Yahoo Finance loaded: {len(snp500_daily)} trading days")

## 2.2 ETF Data - Exchange Traded Funds

ETFs are popular investment vehicles that track indices, sectors, or themes. Let's explore how to download and analyze ETF data.

### Understanding Dividends and Adjusted Close
- **Close Price**: The actual closing price on that day
- **Adjusted Close**: Close price adjusted for dividends and splits
- Always use Adjusted Close for return calculations!

In [ ]:
# VOO - Vanguard S&P 500 ETF
# One of the most popular ETFs tracking the S&P 500
ticker_obj = yf.Ticker("VOO")
voo_etf = ticker_obj.history(start = start, interval = "1d")

print(f"VOO ETF data loaded: {len(voo_etf)} trading days")
print(f"Expense Ratio: 0.03% (very low!)")

In [ ]:
voo_etf.tail()

In [ ]:
# Let's look at an international ETF
# EPI - WisdomTree India Earnings Fund
ticker_obj = yf.Ticker("EPI")
epi_etf_daily = ticker_obj.history(start = start, interval = "1d")

print(f"EPI ETF data loaded: {len(epi_etf_daily)} trading days")

In [ ]:
# Find dividend payments
dividend_days = epi_etf_daily[epi_etf_daily.Dividends > 0]
print(f"Number of dividend payments: {len(dividend_days)}")
print("\nRecent dividends:")
dividend_days.tail()

In [ ]:
# Compare Close vs Adjusted Close to see dividend impact
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# Plot Close and Adjusted Close
ax1.plot(epi_etf_daily.index, epi_etf_daily['Close'], label='Close Price', alpha=0.7)
ax1.plot(epi_etf_daily.index, epi_etf_daily['Close'], label='Adjusted Close', alpha=0.7)
ax1.set_title("EPI ETF: Close vs Adjusted Close")
ax1.set_ylabel("Price ($)")
ax1.legend()
ax1.grid(True)

# Plot dividends
ax2.bar(dividend_days.index, dividend_days['Dividends'], width=20, alpha=0.7)
ax2.set_title("Dividend Payments")
ax2.set_ylabel("Dividend per Share ($)")
ax2.set_xlabel("Date")
ax2.grid(True)

plt.tight_layout()
plt.show()

### International Stock Example

Different exchanges use different ticker symbols:
- US stocks: Just the ticker (e.g., "AAPL")
- Indian stocks: Ticker + ".NS" for NSE (e.g., "RELIANCE.NS")
- German stocks: Ticker + ".DE" (e.g., "BMW.DE")
- UK stocks: Ticker + ".L" for London (e.g., "BP.L")

In [ ]:
# Example: Indian stock from NSE (National Stock Exchange)
# Eicher Motors - Royal Enfield manufacturer
EICHERMOT = yf.download(tickers = "EICHERMOT.NS",
                     period = "max",
                     interval = "1d")

print(f"\nNote: Prices are in Indian Rupees (INR)")
print(f"Current USD/INR rate: ~83")

## 2.3 Paid Data Sources

While Yahoo Finance is great for daily data, paid services offer:
- Real-time data
- Intraday/minute data
- More history
- News feeds
- Fundamental data

### Popular Paid Data Providers

1. **Polygon.io**
   - Free tier: 5 API calls/minute
   - Excellent for news data
   - [Documentation](https://polygon.io/docs/stocks/get_v2_reference_news)

2. **Alpha Vantage**
   - Free tier: 25 API calls/day
   - Good fundamental data
   - [Documentation](https://www.alphavantage.co/documentation/)

3. **Other Options**
   - Quandl (now part of Nasdaq)
   - IEX Cloud
   - Tiingo

📚 **Recommended Articles**:
- [Financial News Summarization](https://pythoninvest.com/long-read/chatgpt-api-for-financial-news-summarization)
- [Stock Screening with Paid Data](https://pythoninvest.com/long-read/stock-screening-using-paid-data)

## 2.4 More Macroeconomic Indicators

Let's explore some additional economic indicators that can impact stock markets:
- Gold reserves and volatility
- Oil prices (WTI and Brent)
- Currency exchange rates

In [ ]:
# Gold reserves excluding gold for China
# Shows China's foreign currency reserves
gold_reserves = pdr.DataReader("TRESEGCNM052N", "fred", start=start)
print(f"China's reserves data loaded: {len(gold_reserves)} months")

In [ ]:
# CBOE Gold ETF Volatility Index
# Measures expected volatility in gold prices
gold_volatility = pdr.DataReader("GVZCLS", "fred", start=start)
print(f"Gold volatility data loaded: {len(gold_volatility)} days")

In [ ]:
# Oil prices - crucial for energy sector and inflation
# WTI (West Texas Intermediate) - US benchmark
oil_wti = pdr.DataReader("DCOILWTICO", "fred", start=start)

# Brent - International benchmark
oil_brent = pdr.DataReader("DCOILBRENTEU", "fred", start=start)

print(f"Current WTI Oil Price: ${oil_wti['DCOILWTICO'].iloc[-1]:.2f}")
print(f"Current Brent Oil Price: ${oil_brent['DCOILBRENTEU'].iloc[-1]:.2f}")

In [ ]:
# Compare WTI and Brent oil prices
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(oil_wti.index, oil_wti['DCOILWTICO'], label='WTI (US)', alpha=0.8)
ax.plot(oil_brent.index, oil_brent['DCOILBRENTEU'], label='Brent (International)', alpha=0.8)

ax.set_xlabel('Date')
ax.set_ylabel('Price ($/barrel)')
ax.set_title('Oil Prices: WTI vs Brent')
ax.legend()
ax.grid(True)

# Highlight negative oil prices in 2020
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.annotate('Negative WTI prices!', xy=('2020-04-20', -20), 
            xytext=('2019-01-01', -30),
            arrowprops=dict(arrowstyle='->', color='red'))

plt.show()

### Web Scraping for Economic Data

Sometimes you need data that's not available through APIs. Let's scrape economic indicators from TradingEconomics.

In [ ]:
# Web scraping example - US economic indicators
# Note: Always check robots.txt and terms of service!

url = "https://tradingeconomics.com/united-states/indicators"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.get(url, headers=headers)
print(f"Response status: {response.status_code}")

In [ ]:
# Parse the webpage if successful
if response.status_code == 200:
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Find the table with economic indicators
    table_div = soup.find("div", class_="table-responsive")
    
    if table_div:
        table = table_div.find("table")
        df = pd.read_html(str(table))[0]
        
        print("\nKey US Economic Indicators:")
        print(df.head(10))
    else:
        print("Table not found - website structure may have changed")
else:
    print("Failed to retrieve data from the webpage.")

## 2.5) Financial Reporting - Company Fundamentals

Fundamental analysis looks at a company's financial health through:
- Income statements (revenue, profits)
- Balance sheets (assets, liabilities)
- Cash flow statements
- Key ratios (P/E, P/B, ROE)

Let's examine NVIDIA as an example:

In [ ]:
# Get NVIDIA's fundamental data
nvda = yf.Ticker('NVDA')

# Basic information
info = nvda.basic_info
print("NVIDIA Corporation (NVDA)")
print(f"Market Cap: ${info.get('marketCap', 0)/1e9:.1f}B")
print(f"P/E Ratio: {info.get('trailingPE', 'N/A')}")
print(f"52-Week High: ${info.get('fiftyTwoWeekHigh', 'N/A')}")
print(f"52-Week Low: ${info.get('fiftyTwoWeekLow', 'N/A')}")

In [ ]:
# Get financial statements
# Income statement (yearly)
print("\nIncome Statement (Last 4 Years):")
nvda.financials

In [ ]:
# Calculate some key metrics from financials
financials = nvda.financials

if not financials.empty:
    # Revenue growth
    revenue = financials.loc['Total Revenue']
    revenue_growth = (revenue.iloc[0] / revenue.iloc[-1]) ** (1/(len(revenue)-1)) - 1
    
    print(f"\nRevenue CAGR: {revenue_growth:.1%}")
    print(f"Latest Revenue: ${revenue.iloc[0]/1e9:.1f}B")
    print(f"Revenue 4 years ago: ${revenue.iloc[-1]/1e9:.1f}B")
    
    # Profit margins
    net_income = financials.loc['Net Income']
    profit_margin = net_income / revenue
    
    print(f"\nLatest Profit Margin: {profit_margin.iloc[0]:.1%}")
    print(f"Profit Margin 4 years ago: {profit_margin.iloc[-1]:.1%}")

## 2.6 Web Scraping - Market Cap Data

Let's scrape a list of companies by market cap to find investment opportunities:

In [ ]:
# Scrape global market cap data
url = "https://companiesmarketcap.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Find the download link
    download_link = soup.find("a", {"rel": "nofollow", "href": "?download=csv"})
    
    if download_link:
        download_url = 'https://companiesmarketcap.com/' + download_link["href"]
        download_response = requests.get(download_url, headers=headers)
        
        if download_response.status_code == 200:
            with open("global_stocks.csv", "wb") as f:
                f.write(download_response.content)
            print("Global market cap data downloaded successfully.")
        else:
            print("Failed to download the CSV file.")
    else:
        print("Download link not found on the webpage.")
else:
    print("Failed to retrieve data from the webpage.")

In [ ]:
# Load and analyze the market cap data
try:
    global_stocks = pd.read_csv("global_stocks.csv")
    global_stocks['marketcap_b_usd'] = global_stocks.marketcap/1e9
    
    print("\nTop 10 Companies by Market Cap:")
    print(global_stocks[['Rank', 'Name', 'Symbol', 'marketcap_b_usd', 'country']].head(10))
    
    # Analysis by country
    print("\n\nTop Countries by Total Market Cap:")
    country_marketcap = global_stocks.groupby('country')['marketcap_b_usd'].sum().sort_values(ascending=False)
    print(country_marketcap.head(10))
    
except FileNotFoundError:
    print("CSV file not found. Web scraping may have failed.")

## 📊 Stock Screeners - Finding Investment Opportunities

### What is a Stock Screener?
A stock screener helps you filter thousands of stocks based on specific criteria to find potential investments.

### Example Screening Criteria
Let's say we want to find:
- S&P 500 companies only (large, stable)
- Dividend yield > 2% (income generation)
- Revenue growth > 0% YoY (growing business)
- PEG ratio < 0.5 (potentially undervalued)
- Earnings next week (potential catalyst)

### Popular Free Screeners
- [TradingView Screener](https://www.tradingview.com/screener/)
- [Yahoo Finance Screener](https://finance.yahoo.com/screener/)
- [Finviz](https://finviz.com/screener.ashx)

📚 **Recommended Reading**: [Stock Screening Using Paid Data](https://pythoninvest.com/long-read/stock-screening-using-paid-data)

# 📋 Project Development Cheat Sheet

## Step-by-Step Guide for Your Capstone Project

### 1️⃣ Select Your Market
- Choose one country/region (e.g., US, India, Germany)
- Or go global (harder but more comprehensive)
- Consider your local knowledge advantage

### 2️⃣ Choose Benchmarks
- Index (S&P 500, DAX, Nifty)
- Risk-free rate (treasury bonds)
- 60/40 portfolio
- Sector ETF

### 3️⃣ Select Macro Indicators
- Interest rates
- Inflation (CPI)
- GDP growth
- Oil prices
- Currency rates
- Sector-specific indicators

### 4️⃣ Define Dataset Size
- **Recommended**: 25 years of data
- **Minimum**: 1 million rows total
- Include 100+ stocks for diversification
- Daily frequency (or higher)

### 5️⃣ Fundamental Data Decision
- Free: Yahoo Finance (limited to 4 years)
- Paid: Alpha Vantage, Polygon.io
- Web scraping: SEC EDGAR, company websites

### 6️⃣ Alternative Data (Optional)
- News sentiment
- Social media mentions
- Google trends
- Weather data (for commodities)
- Satellite imagery

### ⚠️ CRITICAL: Avoid Data Leakage!
**Never use future information in historical predictions:**
- Only use data available at prediction time
- Earnings reported in Q1 2024 → available from April 2024
- Be careful with restatements and revisions
- Use proper time delays for all features

# 📝 Homework Module 1 - Key Concepts

The homework will test your understanding of:

1. **S&P 500 Analysis**
   - Historical performance
   - Market corrections
   - Recovery periods

2. **Data Manipulation**
   - Working with time series
   - Calculating returns
   - Finding patterns

3. **Project Planning**
   - Choosing your market
   - Selecting data sources
   - Defining success metrics

**Deadline**: 2 weeks from lecture date

**Extra Points**: Share your solution on social media and tag @PythonInvest and @DataTalksClub!

# 🎯 Summary and Recommendations

## Key Takeaways

1. **Economic Context Matters**
   - Understand GDP, inflation, and interest rates
   - These drive all investment returns

2. **Data is Everywhere**
   - Start with free sources (Yahoo Finance, FRED)
   - Add paid data as needed
   - Don't forget alternative data

3. **Risk and Return Go Together**
   - Higher returns require higher risk
   - Define your risk tolerance
   - Always compare to benchmarks

4. **Start Simple, Build Complexity**
   - Begin with one stock or index
   - Add features gradually
   - Test everything thoroughly

## Generic Course Recommendations

✅ **Do:**
- Be active in Slack/Telegram
- Start homework early
- Work on project weekly
- Leverage your unique edge

❌ **Don't:**
- Wait until deadlines
- Try to build everything at once
- Ignore data quality issues
- Leak future data

## Next Steps

1. Complete the setup and run all code in this notebook
2. Start Homework 1 immediately
3. Think about your capstone project idea
4. Join the course community channels

**Remember**: This is a marathon, not a sprint. Consistent daily progress beats cramming!

## 🚀 Happy Learning and Investing!

---

*Disclaimer: This is for educational purposes only. Always do your own research before making investment decisions.*